# From SDTM to the primary efficacy table

**Data kit lab** `cdisc-pilot-primary-efficacy` · Clinical · about 60 minutes

**The question.** In study CDISCPILOT01, does the change in ADAS-Cog (11) from baseline to Week 24 show a dose response across placebo, low-dose and high-dose xanomeline?

**What you do.** You rebuild Table 14-3.01 of the study report from the public datasets and check every number.

1. Open the tabulation data (SDTM DM and QS).
2. Open the analysis data (ADaM ADSL and ADQSADAS) and trace it back to SDTM.
3. Select the analysis set the report used.
4. Compute the summary statistics and fit the model.
5. Compare your numbers with the published table.

**The data.** CDISC SDTM/ADaM Pilot Project, updated pilot submission package, from the CDISC repository on GitHub. The data are CDISC's and are used here under CDISC's terms of use. Nothing is copied into this notebook: the first cell downloads the files from the source (about 39 MB, once).

**You need.** Python 3.10 or later with `pandas` and `statsmodels`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'cdisc-pilot-primary-efficacy'
FILES = json.loads(r'''{
 "dm": {
  "url": "https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm/dm.xpt",
  "name": "dm.xpt",
  "sha256": "7327baea97fd532d02385248da0c7240402e770099507e2c3a88e2ac706c02a6",
  "bytes": 110800
 },
 "qs": {
  "url": "https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm/qs.xpt",
  "name": "qs.xpt",
  "sha256": "a1a68fbebbceb6237a8fba0f96b6b9ab8d476817200cde5a0b625966ac23cd6e",
  "bytes": 33849760
 },
 "adsl": {
  "url": "https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/analysis/adam/datasets/adsl.xpt",
  "name": "adsl.xpt",
  "sha256": "c5139f873a93ef6add77bc4297beb6460a398bbdd036874382a9f39dfc92091d",
  "bytes": 114640
 },
 "adqsadas": {
  "url": "https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/analysis/adam/datasets/adqsadas.xpt",
  "name": "adqsadas.xpt",
  "sha256": "ac3fa1217651810b0f15fe489fdda7d99eff10553ded025156ad7c1b89f84608",
  "bytes": 5178480
 }
}''')
PUBLISHED = json.loads(r'''{
 "baseline_mean_placebo": {
  "label": "Baseline mean, Placebo",
  "value": 24.1,
  "tolerance": {
   "abs": 0.05
  }
 },
 "baseline_sd_placebo": {
  "label": "Baseline SD, Placebo",
  "value": 12.19,
  "tolerance": {
   "abs": 0.005
  }
 },
 "baseline_mean_low": {
  "label": "Baseline mean, Xanomeline Low Dose",
  "value": 24.4,
  "tolerance": {
   "abs": 0.05
  }
 },
 "baseline_sd_low": {
  "label": "Baseline SD, Xanomeline Low Dose",
  "value": 12.92,
  "tolerance": {
   "abs": 0.005
  }
 },
 "baseline_mean_high": {
  "label": "Baseline mean, Xanomeline High Dose",
  "value": 21.3,
  "tolerance": {
   "abs": 0.05
  }
 },
 "baseline_sd_high": {
  "label": "Baseline SD, Xanomeline High Dose",
  "value": 11.74,
  "tolerance": {
   "abs": 0.005
  }
 },
 "week24_mean_placebo": {
  "label": "Week 24 mean, Placebo",
  "value": 26.7,
  "tolerance": {
   "abs": 0.05
  }
 },
 "week24_sd_placebo": {
  "label": "Week 24 SD, Placebo",
  "value": 13.79,
  "tolerance": {
   "abs": 0.005
  }
 },
 "week24_mean_low": {
  "label": "Week 24 mean, Xanomeline Low Dose",
  "value": 26.4,
  "tolerance": {
   "abs": 0.05
  }
 },
 "week24_sd_low": {
  "label": "Week 24 SD, Xanomeline Low Dose",
  "value": 13.18,
  "tolerance": {
   "abs": 0.005
  }
 },
 "week24_mean_high": {
  "label": "Week 24 mean, Xanomeline High Dose",
  "value": 22.8,
  "tolerance": {
   "abs": 0.05
  }
 },
 "week24_sd_high": {
  "label": "Week 24 SD, Xanomeline High Dose",
  "value": 12.48,
  "tolerance": {
   "abs": 0.005
  }
 },
 "n_placebo": {
  "label": "n, Placebo",
  "value": 79,
  "tolerance": {
   "exact": true
  }
 },
 "change_mean_placebo": {
  "label": "Change from baseline mean, Placebo",
  "value": 2.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "change_sd_placebo": {
  "label": "Change from baseline SD, Placebo",
  "value": 5.8,
  "tolerance": {
   "abs": 0.005
  }
 },
 "n_low": {
  "label": "n, Xanomeline Low Dose",
  "value": 81,
  "tolerance": {
   "exact": true
  }
 },
 "change_mean_low": {
  "label": "Change from baseline mean, Xanomeline Low Dose",
  "value": 2.0,
  "tolerance": {
   "abs": 0.05
  }
 },
 "change_sd_low": {
  "label": "Change from baseline SD, Xanomeline Low Dose",
  "value": 5.55,
  "tolerance": {
   "abs": 0.005
  }
 },
 "n_high": {
  "label": "n, Xanomeline High Dose",
  "value": 74,
  "tolerance": {
   "exact": true
  }
 },
 "change_mean_high": {
  "label": "Change from baseline mean, Xanomeline High Dose",
  "value": 1.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "change_sd_high": {
  "label": "Change from baseline SD, Xanomeline High Dose",
  "value": 4.26,
  "tolerance": {
   "abs": 0.005
  }
 },
 "p_dose_response": {
  "label": "p-value, dose response",
  "value": 0.245,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "p_low_vs_placebo": {
  "label": "p-value, low dose minus placebo",
  "value": 0.569,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "p_high_vs_placebo": {
  "label": "p-value, high dose minus placebo",
  "value": 0.233,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "p_high_vs_low": {
  "label": "p-value, high dose minus low dose",
  "value": 0.52,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "lsdiff_low_vs_placebo": {
  "label": "LS mean difference, low dose minus placebo",
  "value": -0.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "se_low_vs_placebo": {
  "label": "SE of difference, low dose minus placebo",
  "value": 0.82,
  "tolerance": {
   "abs": 0.005
  }
 },
 "lsdiff_high_vs_placebo": {
  "label": "LS mean difference, high dose minus placebo",
  "value": -1.0,
  "tolerance": {
   "abs": 0.05
  }
 },
 "se_high_vs_placebo": {
  "label": "SE of difference, high dose minus placebo",
  "value": 0.84,
  "tolerance": {
   "abs": 0.005
  }
 },
 "lsdiff_high_vs_low": {
  "label": "LS mean difference, high dose minus low dose",
  "value": -0.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "se_high_vs_low": {
  "label": "SE of difference, high dose minus low dose",
  "value": 0.84,
  "tolerance": {
   "abs": 0.005
  }
 }
}''')
PUBLISHED_FROM = 'Table 14-3.01, Primary Endpoint Analysis: ADAS Cog (11) - Change from Baseline to Week 24 - LOCF, pdf page 468 | https://raw.githubusercontent.com/cdisc-org/sdtm-adam-pilot-project/master/updated-pilot-submission-package/900172/m5/53-clin-stud-rep/535-rep-effic-safety-stud/5351-stud-rep-contr/cdiscpilot01/cdiscpilot01.pdf'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Who is in the study? (SDTM DM and ADaM ADSL)

The files are in SAS transport format (XPT). `pandas` reads it.

Look at the raw read first. The planned-treatment number for placebo should be 0.

In [ ]:
import pandas as pd

raw = pd.read_sas(data_file("adqsadas"), format="xport", encoding="utf-8")
print(raw[["TRTP", "TRTPN"]].drop_duplicates().to_string(index=False))
print("rows where TRTPN == 0:", int((raw["TRTPN"] == 0).sum()))

If you see `5.4e-79` and not `0`, you have met the first trap. With this reader, a zero stored in the transport file comes back as a tiny number. A filter such as `TRTPN == 0` then finds nothing.

The fix below sets those values to 0 and fills empty text with `""`. Use `read_xpt` for every file from here on.

In [ ]:
def read_xpt(file_id):
    """Read one dataset file and repair the two reading problems."""
    df = pd.read_sas(data_file(file_id), format="xport", encoding="utf-8")
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].fillna("")
        elif df[col].dtype.kind == "f":
            df[col] = df[col].where(df[col].abs() > 1e-70, 0.0)
    return df


dm = read_xpt("dm")
adsl = read_xpt("adsl")
print("DM rows:", len(dm), "| one row per subject:", dm["USUBJID"].is_unique)
print(dm["ARM"].value_counts().to_string())
print()
print("ADSL rows:", len(adsl))
flags = adsl.groupby("TRT01P")[["ITTFL", "SAFFL", "EFFFL"]].agg(lambda s: int((s == "Y").sum()))
print(flags.to_string())

DM has more rows than ADSL because DM also holds the screen failures. ADSL has the subjects assigned to a treatment arm and one flag per analysis population.

The table you are rebuilding says "Population: Efficacy". Its column headers should equal the `EFFFL` counts above.

## 2. The scores as collected (SDTM QS)

QS is a long table: one row per subject, question and visit. The ADAS-Cog (11) total has the test code `ACTOT`.

In [ ]:
qs = read_xpt("qs")
print("QS rows:", len(qs))
qs_total = qs[qs["QSTESTCD"] == "ACTOT"]
print("test name:", qs_total["QSTEST"].unique().tolist(), "| rows:", len(qs_total))
print(qs_total["VISIT"].value_counts().to_string())
print()
one = qs_total[qs_total["USUBJID"] == "01-701-1015"]
print(one[["USUBJID", "QSSEQ", "VISIT", "QSDTC", "QSSTRESN"]].to_string(index=False))

Note the visits. Besides Baseline and Weeks 8, 16 and 24 there are unscheduled and early-termination visits. The analysis dataset has to decide which record counts for which analysis visit.

## 3. The analysis dataset (ADaM ADQSADAS) and the trace back to SDTM

ADQSADAS has one row per subject, parameter and analysis visit. The data guide says two things you can test:

- records with `DTYPE` equal to `LOCF` were filled in by carrying the last score forward;
- every other record traces back to SDTM QS by `USUBJID` and `QSSEQ`.

In [ ]:
adas = read_xpt("adqsadas")
total = adas[adas["PARAMCD"] == "ACTOT"]
print(total["AVISIT"].value_counts().to_string())
print()
print("DTYPE:", total["DTYPE"].replace("", "(observed)").value_counts().to_dict())

observed = total[total["DTYPE"] == ""]
trace = observed.merge(qs_total[["USUBJID", "QSSEQ", "QSSTRESN"]], on=["USUBJID", "QSSEQ"], how="left")
n_same = int((trace["AVAL"] == trace["QSSTRESN"]).sum())
print(f"{len(observed)} observed analysis values; {n_same} equal the SDTM value they point to")

If the two numbers are equal, every observed analysis value is the collected value. That is what traceability means in practice: you can follow a number in the analysis back to the record it came from.

## 4. Select the analysis set

The report's synopsis says the primary analysis used the efficacy population, with LOCF for a missing Week 24 value.

So keep: the total score, analysis visit Week 24, efficacy population, and the record flagged for analysis.

In [ ]:
wk24 = total[(total["AVISITN"] == 24) & (total["EFFFL"] == "Y") & (total["ANL01FL"] == "Y")].copy()
print("rows:", len(wk24), "| subjects:", wk24["USUBJID"].nunique())
print("carried forward (LOCF):", int((wk24["DTYPE"] == "LOCF").sum()), "of", len(wk24))

# The file header carries a label for every variable. Read the labels of the three columns you are about to use.
reader = pd.read_sas(data_file("adqsadas"), format="xport", encoding="utf-8", iterator=True)
as_text = lambda x: (x.decode() if isinstance(x, bytes) else x).strip()
labels = {as_text(f["name"]): as_text(f["label"]) for f in reader.fields}
reader.close()
for col in ("AVAL", "BASE", "CHG"):
    print(f"{col:<5} label in the file: {labels[col]}")
print("CHG equals AVAL - BASE on every row:", bool(((wk24["AVAL"] - wk24["BASE"] - wk24["CHG"]).abs() < 1e-9).all()))

One row per subject: good. Count the carried-forward values before you read any result. They are a large share of this analysis.

Look at the label of `CHG`. It is the same as the label of `BASE`, which is wrong: the values show that `CHG` is the change from baseline. Trust a variable's values and the define file over its label.


## 5. Summary statistics

Baseline, Week 24 and change from baseline, by planned treatment.

In [ ]:
ARMS = {"placebo": "Placebo", "low": "Xanomeline Low Dose", "high": "Xanomeline High Dose"}
by_arm = wk24.groupby("TRTP")
summary = pd.DataFrame({
    "n": by_arm.size(),
    "baseline_mean": by_arm["BASE"].mean(), "baseline_sd": by_arm["BASE"].std(),
    "week24_mean": by_arm["AVAL"].mean(), "week24_sd": by_arm["AVAL"].std(),
    "change_mean": by_arm["CHG"].mean(), "change_sd": by_arm["CHG"].std(),
}).loc[list(ARMS.values())]
print(summary.round(2).to_string())

## 6. The model

The table's footnotes give the model: analysis of covariance with treatment and site group as factors and the baseline value as a covariate.

- **Dose response:** treatment enters as a number (the dose: 0, 54 or 81) and you test whether its coefficient is zero.
- **Pairwise comparisons:** treatment enters as a category. The p-values are not adjusted for multiple comparisons.

With no interaction terms, the difference between two adjusted (least-squares) means is the difference between the two treatment coefficients.

In [ ]:
import statsmodels.formula.api as smf

dose = smf.ols("CHG ~ TRTPN + C(SITEGR1) + BASE", data=wk24).fit()
print("dose-response p-value:", round(dose.pvalues["TRTPN"], 4))

wk24["ARM"] = pd.Categorical(wk24["TRTP"], categories=list(ARMS.values()))
pair = smf.ols("CHG ~ C(ARM) + C(SITEGR1) + BASE", data=wk24).fit()
LOW, HIGH = "C(ARM)[T.Xanomeline Low Dose]", "C(ARM)[T.Xanomeline High Dose]"
high_low = pair.t_test(f"{HIGH} - {LOW} = 0")
for name, est, se, p in [
    ("low - placebo", pair.params[LOW], pair.bse[LOW], pair.pvalues[LOW]),
    ("high - placebo", pair.params[HIGH], pair.bse[HIGH], pair.pvalues[HIGH]),
    ("high - low", float(high_low.effect[0]), float(high_low.sd[0][0]), float(high_low.pvalue)),
]:
    print(f"{name:<15} difference {est:6.2f}  SE {se:.2f}  p {p:.3f}")

## 7. Check against the published table

`PUBLISHED` holds the values copied by code from Table 14-3.01 of the study report (PDF page 468). `report` prints your value beside each one and says whether it is inside the tolerance: half a unit in the last decimal place the report prints.

In [ ]:
RESULTS = {}
for key, arm in ARMS.items():
    row = summary.loc[arm]
    RESULTS[f"n_{key}"] = int(row["n"])
    for stat in ("baseline_mean", "baseline_sd", "week24_mean", "week24_sd", "change_mean", "change_sd"):
        RESULTS[f"{stat}_{key}"] = float(row[stat])
RESULTS.update({
    "p_dose_response": float(dose.pvalues["TRTPN"]),
    "p_low_vs_placebo": float(pair.pvalues[LOW]),
    "p_high_vs_placebo": float(pair.pvalues[HIGH]),
    "p_high_vs_low": float(high_low.pvalue),
    "lsdiff_low_vs_placebo": float(pair.params[LOW]),
    "se_low_vs_placebo": float(pair.bse[LOW]),
    "lsdiff_high_vs_placebo": float(pair.params[HIGH]),
    "se_high_vs_placebo": float(pair.bse[HIGH]),
    "lsdiff_high_vs_low": float(high_low.effect[0]),
    "se_high_vs_low": float(high_low.sd[0][0]),
})
report(RESULTS)

## What the result says

Read the dose-response p-value against the usual 0.05 level, then read the report's own conclusion in its synopsis. Do they agree?

## Try next

1. Drop the carried-forward rows (`DTYPE == "LOCF"`) and repeat sections 5 and 6. How do n, the means and the p-value move?
2. Repeat the analysis at Week 8 and compare with Table 14-3.03 of the report.
3. Remove the zero repair from `read_xpt` and rerun. Which steps still work, and which silently change?
4. In QS, find a subject with a "RETRIEVAL" or unscheduled visit. Which analysis visit did that record get in ADQSADAS?